# Studio 2 — Kelompok 06 · Training di Google Colab

Sebelum menjalankan:
1. Upload `train.zip` ke Google Drive, di folder `MyDrive/sampah-studio2/`.
2. **Runtime → Change runtime type → T4 GPU** → Save.

Hasil (`model.pth` + `experiments/`) otomatis disalin ke `MyDrive/sampah-studio2/hasil/`,
jadi tetap aman walaupun sesi Colab terputus.

In [ ]:
# Sambungkan Google Drive (akan muncul jendela izin akun Google)
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/sampah-studio2'  # ubah jika folder di Drive berbeda
!ls -lh {DRIVE_DIR}

In [ ]:
# Salin dataset ke disk lokal Colab lalu ekstrak (membaca langsung dari Drive jauh lebih lambat)
!cp {DRIVE_DIR}/train.zip /content/train.zip
!unzip -q -o /content/train.zip -d /content/data
DATA = '/content/data/train'
!for d in {DATA}/*/; do echo "$d $(ls $d | wc -l)"; done

In [ ]:
!git clone https://github.com/beningapniprameswari/Studio-2_Kelompok-6.git /content/Studio-2_Kelompok-6
%cd /content/Studio-2_Kelompok-6
!pip install -q -r submission/requirements.txt
!mkdir -p experiments
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Baseline ResNet-18 (pembanding, opsional)
Bobot disimpan ke `baseline/model.pth`, tidak menimpa submission utama.

In [ ]:
!python train_baseline.py --data {DATA} --epochs 8 2>&1 | tee experiments/01_baseline_resnet18.txt
!python evaluate.py --submission ./baseline --data ./my_val 2>&1 | tee -a experiments/01_baseline_resnet18.txt

## 2. ConvNeXt-Tiny (submission utama)
Latih di 85% data, pilih epoch terbaik dengan Macro-F1 `my_val/` (split yang sama dengan baseline).

In [ ]:
!python train.py --data {DATA} --epochs 15 --workers 2 2>&1 | tee experiments/02_convnext_tiny.txt

In [ ]:
# Uji submission persis seperti asisten (evaluate.py resmi), lalu simpan hasil ke Drive
!python evaluate.py --submission ./submission --data ./my_val 2>&1 | tee -a experiments/02_convnext_tiny.txt
!ls -lh submission/model.pth
!mkdir -p {DRIVE_DIR}/hasil
!cp submission/model.pth {DRIVE_DIR}/hasil/model.pth
!cp -r experiments {DRIVE_DIR}/hasil/
!ls -lhR {DRIVE_DIR}/hasil